In [34]:
# INITIALIZATION
import mpmath as mp
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import natural_units as nu
from numpy.polynomial.laguerre import laggauss
mp.mp.dps = 25
# '_fid' parameters are in natural units, 'my_' parameters are remormalized by fids.
rho_s     = 2.74e8 * nu.mSun / nu.kpc**3
r_s       = 0.141 * nu.kpc
sigma_fid = 1 / rho_s / r_s
v_fid     = mp.sqrt(4 * mp.pi * nu.G_Newton * rho_s) * r_s
lumi_fid  = mp.power(4 * mp.pi * rho_s * r_s**2, 5/2) * mp.power(nu.G_Newton, 3/2)
t_fid     = 1 / mp.sqrt(4 * mp.pi * nu.G_Newton * rho_s)
C_fid     = mp.power(4 * mp.pi * nu.G_Newton, 3/2) * mp.power(rho_s, 5/2) * r_s**2
t_fid_in_gyr = t_fid / (1e9 * nu.year)
# Model parameters
a = mp.mpf('2.257')
c = mp.mpf('0.75')
my_mass_norm = mp.mpf('0.0') # M_b/(4*pi*rho_s*r_s^3)
my_scale_norm = mp.mpf('0.1') # normalized baryon scale radius, a/r_s
# The following are all the velocity-dependent parameters.
m_chi = 1 * nu.GeV
m_phi = 1e-2 * nu.keV
omega = m_phi / m_chi
# omega as a velocity also needs to be converted
my_omega = omega / v_fid
# sigma_0 takes a 1/m to be in the form of sigma/m like SIDM strength
g_chi = 1e-6
sigma_0 = g_chi**4 / 4 / mp.pi / m_chi**2 / omega**4 / m_chi
# sigma_0 = 24000 * nu.cm**2 / nu.gram
my_sigma_0 = sigma_0 / sigma_fid
print('sigma_0 = ' + str(sigma_0 / (nu.cm**2 / nu.gram)) + ' cm^2 per gram')

sigma_0 = 1738.177777987486856199841 cm^2 per gram


In [ ]:
# MAJOR FUNCTIONS
def density_dm(r):
    """Dark matter density function, assuming a NFW profile"""
    return 1/(r * (1+r)**2)

def mass_dm(r):
    """Dark matter mass function, assuming a NFW profile"""
    return -r/(1+r) + mp.log(1+r)

def density_baryon(r, mass_norm, ars):
    """Baryon density function, assuming a Plummer profile"""
    """mass_norm is the normalized baryon mass, M_b/(4*pi*rho_s*r_s^3)"""
    """ars is the normalized scale radius, a/r_s"""
    return (3*mass_norm)/(ars**3) * (1+r**2/ars**2)**(-5/2)

def mass_baryon(r, mass_norm, ars):
    """Baryon mass function, assuming a Plummer profile"""
    return mass_norm * (1 + ars**2 * r**(-2))**(-1.5)

def density_total(r, mass_norm, ars):
    """Total density function"""
    return density_dm(r) + density_baryon(r, mass_norm, ars)

def mass_total(r, mass_norm, ars):
    """Total enclosed mass function"""
    return mass_dm(r) + mass_baryon(r, mass_norm, ars)

def vd_dm(r, mass_norm, ars):
    """Dark matter 1D velocity dispersion"""
    r, mass_norm, ars = mp.mpf(r), mp.mpf(mass_norm), mp.mpf(ars)
    
    term = -(1/2) * r * (
        (2 * mass_norm * (1+r) * (
            mp.sqrt(1+ars**2) * (
                -ars**4 - (1+r) * (-r + mp.sqrt(ars**2 + r**2)) + 
                ars**2 * (2+r-2*r**2 + 2*mp.sqrt(ars**2 + r**2) + 2*r*mp.sqrt(ars**2 + r**2))
            ) - 
            6 * ars**2 * (1+r) * mp.sqrt(ars**2 + r**2) * (
                mp.acoth(mp.sqrt(1+ars**2)) + 
                mp.atanh((-1-r+mp.sqrt(ars**2 + r**2))/mp.sqrt(1+ars**2))
            )
        )) / (ars**2 * (1+ars**2)**(5/2) * mp.sqrt(ars**2 + r**2))
        +
        (2 + 9*r + 6*r**2 - 6*r * (1+r)**2 * mp.log(1+1/r)) / r
        +
        (1/r**2) * (1+r) * (
            -r * (1 + r*(-1 + mp.pi**2 * (1+r)) + 5*r*(1+r)*mp.log(r)) + 
            (-1 + r*(3 + r*(11 + 5*r))) * mp.log(1+r) - 
            3 * r**2 * (1+r) * mp.log(1+r)**2 - 
            6 * r**2 * (1+r) * mp.polylog(2, -r)
        )
    )
    
    # Ensure we get a real result (in case of small numerical errors)
    if mp.im(term) != 0 and abs(mp.im(term)) < 1e-10:
        term = mp.re(term)
    
    result = mp.sqrt(term)
    return result

def big_dev(r, mass_norm, ars):
    """deviation function of partial vd_dm^2/partial r"""
    r, mass_norm, ars = mp.mpf(r), mp.mpf(mass_norm), mp.mpf(ars)
    
    result = 1/2 * (
        -4 - (4*mass_norm)/(1+ars**2)**2 + (2*mass_norm)/(ars+ars**3)**2 + mp.pi**2 - 
        4*r - (16*mass_norm*r)/(1+ars**2)**2 + (8*mass_norm*r)/(ars+ars**3)**2 + 4*mp.pi**2*r + 
        5*r**2 - (12*mass_norm*r**2)/(1+ars**2)**2 + (6*mass_norm*r**2)/(ars+ars**3)**2 + 3*mp.pi**2*r**2 - 
        8/(1+r) - (26*r)/(1+r) - (22*r**2)/(1+r) - (5*r**3)/(1+r) + 
        
        (4*mass_norm*r**2)/((1+ars**2)**2 * (ars**2+r**2)**(3/2)) - 
        (2*ars**2*mass_norm*r**2)/((1+ars**2)**2 * (ars**2+r**2)**(3/2)) + 
        (6*mass_norm*r**3)/((1+ars**2)**2 * (ars**2+r**2)**(3/2)) - 
        (2*ars**2*mass_norm*r**3)/((1+ars**2)**2 * (ars**2+r**2)**(3/2)) + 
        (2*mass_norm*r**3)/((ars+ars**3)**2 * (ars**2+r**2)**(3/2)) - 
        (2*mass_norm*r**4)/((1+ars**2)**2 * (ars**2+r**2)**(3/2)) + 
        (4*mass_norm*r**4)/((ars+ars**3)**2 * (ars**2+r**2)**(3/2)) - 
        (4*mass_norm*r**5)/((1+ars**2)**2 * (ars**2+r**2)**(3/2)) + 
        (2*mass_norm*r**5)/((ars+ars**3)**2 * (ars**2+r**2)**(3/2)) - 
        
        (4*mass_norm)/((1+ars**2)**2 * mp.sqrt(ars**2+r**2)) + 
        (2*ars**2*mass_norm)/((1+ars**2)**2 * mp.sqrt(ars**2+r**2)) - 
        (12*mass_norm*r)/((1+ars**2)**2 * mp.sqrt(ars**2+r**2)) + 
        (4*ars**2*mass_norm*r)/((1+ars**2)**2 * mp.sqrt(ars**2+r**2)) - 
        (4*mass_norm*r)/((ars+ars**3)**2 * mp.sqrt(ars**2+r**2)) + 
        (6*mass_norm*r**2)/((1+ars**2)**2 * mp.sqrt(ars**2+r**2)) - 
        (12*mass_norm*r**2)/((ars+ars**3)**2 * mp.sqrt(ars**2+r**2)) + 
        (16*mass_norm*r**3)/((1+ars**2)**2 * mp.sqrt(ars**2+r**2)) - 
        (8*mass_norm*r**3)/((ars+ars**3)**2 * mp.sqrt(ars**2+r**2)) + 
        
        1/(r+r**2) - 
        (6*mass_norm*r)/((1+ars**2)**2 * (1+r) * (-r+mp.sqrt(ars**2+r**2))) - 
        (12*mass_norm*r**2)/((1+ars**2)**2 * (1+r) * (-r+mp.sqrt(ars**2+r**2))) - 
        (6*mass_norm*r**3)/((1+ars**2)**2 * (1+r) * (-r+mp.sqrt(ars**2+r**2))) + 
        
        (6*mass_norm*r**2)/((1+ars**2)**2 * (1+r) * mp.sqrt(ars**2+r**2) * (-r+mp.sqrt(ars**2+r**2))) + 
        (12*mass_norm*r**3)/((1+ars**2)**2 * (1+r) * mp.sqrt(ars**2+r**2) * (-r+mp.sqrt(ars**2+r**2))) + 
        (6*mass_norm*r**4)/((1+ars**2)**2 * (1+r) * mp.sqrt(ars**2+r**2) * (-r+mp.sqrt(ars**2+r**2))) + 
        
        (12*mass_norm * (1+4*r+3*r**2) * mp.acoth(mp.sqrt(1+ars**2)))/(1+ars**2)**(5/2) + 
        (12*mass_norm * mp.atanh((-1-r+mp.sqrt(ars**2+r**2))/mp.sqrt(1+ars**2)))/(1+ars**2)**(5/2) + 
        (48*mass_norm*r * mp.atanh((-1-r+mp.sqrt(ars**2+r**2))/mp.sqrt(1+ars**2)))/(1+ars**2)**(5/2) + 
        (36*mass_norm*r**2 * mp.atanh((-1-r+mp.sqrt(ars**2+r**2))/mp.sqrt(1+ars**2)))/(1+ars**2)**(5/2) + 
        
        6*mp.log(1+1/r) + 24*r*mp.log(1+1/r) + 18*r**2*mp.log(1+1/r) + 
        5*mp.log(r) + 20*r*mp.log(r) + 15*r**2*mp.log(r) - 
        20*mp.log(1+r) - mp.log(1+r)/r**2 - 38*r*mp.log(1+r) - 15*r**2*mp.log(1+r) + 
        3*mp.log(1+r)**2 + 12*r*mp.log(1+r)**2 + 9*r**2*mp.log(1+r)**2 + 
        6 * (1+4*r+3*r**2) * mp.polylog(2, -r)
    )
    
    return result

# We are in place to define particle physics functions.
# differential cross section only takes the dimensionless velocity and angular terms, without the sigma at front.
def diff_cs_ruth(v, w, x): # v for velocity (renormalized), x for cos\theta
    y = v**2 / w**2
    return 1 / 2 / (1 + y * (1 - x) / 2)**2

def diff_cs_moll(v, w, x):
    y = v**2 / w**2
    top  = (3 * x**2 + 1) * y**2 + 4 * y + 4
    down = ( (1 - x**2) * y**2 + 4 * y + 4 )**2
    return top / down

def tot_cs_ruth(v, w): # total cross section but without sigma at front
    v_mp = mp.mpf(v)
    w_mp = mp.mpf(w)
    y = v_mp**2 / w_mp**2
    return 1 / (1+y)

def tot_cs_moll(v, w):
    v_mp = mp.mpf(v)
    w_mp = mp.mpf(w)
    y = v_mp**2 / w_mp**2
    return 1 / (1 + y) - 1 / (y**2 + 2 * y) * mp.log(1 + y)

def I_ruth(v, w): # angular integral of cross section with weight of sin^2(theta)
    v_mp = mp.mpf(v)
    w_mp = mp.mpf(w)
    y = v_mp**2 / w_mp**2
    if abs(y) < mp.mpf(1e-4):
        return (mp.mpf(2/3)
                - mp.mpf(2/3) * y
                + mp.mpf(3/5) * y**2)
    return 4 * ((2 + y) * mp.log(1 + y) - 2 * y) / y**3

def I_moll(v, w):
    v_mp = mp.mpf(v)
    w_mp = mp.mpf(w)
    y = v_mp**2 / w_mp**2
    if abs(y) < mp.mpf(1e-4):
        return (mp.mpf(1/3)
                - mp.mpf(1/3) * y
                + mp.mpf(1/3) * y**2
                - mp.mpf(1/3) * y**3)
    top = 2 * (2 * (y**2 + 5 * y + 5) * mp.log(1 + y) - 5 * (y**2 + 2 * y))
    down = y**3 * (2 + y)
    return top / down

def big_int(vd, w , cs_type):
    vd_mp = mp.mpf(vd)
    w_mp = mp.mpf(w)

    def integrand(x):
        x_mp = mp.mpf(x)
        v_rel = 2 * vd_mp * mp.sqrt(x_mp)
        if cs_type == "ruth":
            Isig = I_ruth(v_rel, w_mp)
        elif cs_type == "moll":
            Isig = I_moll(v_rel, w_mp)
        else:
            raise ValueError(f"Unknown cs_type '{cs_type}'. Use 'ruth' or 'moll'.")
        
        return x_mp**3 * Isig * mp.e**(-x_mp)
    
    # 用 mp.quad 在 [0, ∞) 上积分
    integral_val = mp.quad(integrand, [0, mp.inf])
    return 128 * integral_val

def sigma_eff(vd, w, my_sigma_0, cs_type="ruth"):
    bi = big_int(vd, w, cs_type)
    return my_sigma_0 * bi / 512

# Velocity-Dependent conductivity and lumonosity
def luminosity_dm(r, a, c, my_sigma_0, w, mass_norm, ars, cs_type):
    """Dark matter luminosity function"""
    r_val = mp.mpf(r)
    
    density = density_dm(r_val)
    vd = vd_dm(r_val, mass_norm, ars)
    bd = big_dev(r_val, mass_norm, ars)
    bi = big_int(vd, w, cs_type=cs_type)
    smfp = 600 * mp.sqrt(mp.pi) * vd / my_sigma_0 / bi
    lmfp = 3 / 2 * a * c * density * vd**3 * my_sigma_0 * bi / 512
    return (-1) * r_val**2 * smfp * lmfp / (smfp + lmfp) * bd     # DON'T FORGET THE MINUS SIGN AND THE R SQUARE!!!

def cooling_brem(r_val, mass_norm, ars):   # 注意！！！现在只是形式上编好了，但是各个物理量用的单位尚未统一
    vd = vd_dm(r_val, mass_norm, ars)
    rho = density_dm(r_val)
    # E' * (dsigma' / dE') for the Bremsstrahlung process
    def brem_ecs(v, E):
        x = 4 * E / m_chi / v**2
        y = (m_phi / E)**2
        s = mp.sqrt(1 - x)
        return 1 / 3.0 / mp.pi**3 / v**2 * (1 + y / 2) * g_chi**6 / m_chi**2 * mp.sqrt(1 - y) * 2 * mp.atanh(s)
    
    def inner_int(v_rel):
        Emax = m_chi * v_rel**2 / 4
        if Emax <= m_phi :
            return mp.mpf('0')
        f = lambda Ep: brem_ecs(Ep, v_rel)
        return mp.quad(f, [m_phi, Emax])
    def integrand_v(v_rel):
        I_E = inner_int(v_rel)
        return v_rel**3 * mp.e**(-(v_rel/vd)**2 / 4) * I_E
    v_min = mp.sqrt
    outer = mp.quad(integrand_v, [v_min, mp.inf])

    prefactor = 1 / 2.0 / mp.sqrt(mp.pi) / vd**3 * (1 / 4.0) * (rho / m_chi)**2
    return prefactor * outer
    
# Create logarithmically spaced radius points
def log_space(start, stop, num):
    """Create logarithmically spaced points similar to Mathematica's Subdivide"""
    start_log = mp.log10(start)
    stop_log = mp.log10(stop)
    step = (stop_log - start_log) / (num - 1)
    return [mp.power(10, start_log + i * step) for i in range(num)]

In [36]:
sigma_eff(5.07 * nu.km / nu.sec , omega, sigma_0, "ruth") / (nu.cm**2 / nu.gram)

mpf('1.949019193226246441724962626e-10')